# Page 9 — "Outcome vs Process Supervision": two graders on the same chains

One set of reasoning chains, graded twice. An **outcome** grader reads the final answer. A
**process** grader reads every step. This notebook measures what the second one buys, in three
numbers the page has never been able to state:

1. **Detection** — among chains that reached the *right* answer, how many contain a step that is
   demonstrably false. Outcome grading passes every one of them, and page 12's STaR filter would
   fine-tune on every one of them.
2. **Localisation** — among chains that reached the *wrong* answer, how much of the chain was
   sound up to the break. Outcome grading condemns all of it.
3. **Selection** — a process-aware rung on page 8's ladder, if page 8's inputs are on disk.

**Two process graders, and one of them has to earn its labels.** The arithmetic checker is
deterministic: where it judges it is right by construction, and a reader can check any label by
hand. `Qwen2.5-Math-PRM-7B` is a model, so its labels are opinions until measured. Section 6
measures them against the checker on the steps both can see, reports AUC and specificity rather
than claiming a reliability, and **withholds the PRM's rows entirely if they miss a bar set in
section 2 before the run**. That is the move page 8 makes with its verifiers, with one addition:
a verifier that fails its own measurement does not get a footnote, it gets suppressed.

Detection and localisation are the checker's alone, so the page's two load-bearing numbers never
depend on whether the PRM showed up or how it did.

Writes `out/page_09/`: `traces.jsonl` (chains with per-step labels), `derived.json`,
`manifest.json`.

All rationale lives in [`docs/page_9_refactor.md`](../docs/page_9_refactor.md).

## How to run this

| Mode | Chains | Process grader | Needs |
|---|---|---|---|
| `REPLAY` | page 7's pile, filtered | checker only | no GPU, no weights |
| `QUICK` | 24 generated here | checker only | a laptop CPU, ~30 min |
| `FULL` | 100, page 7's pile if present | checker + `Qwen2.5-Math-PRM-7B` | a 16 GB GPU |

**Which chains this page may grade, and it is a filter rather than an exclusion.** A chain that
was cut off and forced to commit is broken by *us*, so counting that break as the model reasoning
badly would report our own experiment as a finding. Page 6's sweep is capped on purpose and
page 7's *split* axis is budget-starved on purpose, so neither qualifies. Page 7's **pile** does:
those chains run to their natural end, and the ones where the cap never bound are exactly what
this page needs. Section 3 applies the filter, says how many survived it, and measures which
problems it dropped, because that filter is a selection and not a formality.

**Two sources, two decoding regimes, and the manifest has to name which one ran.** Page 7's pile
was sampled at the temperature page 7 chose, so a reused chain is one draw from a distribution.
The chains this notebook generates for itself are greedy and carry a step-per-line instruction
page 7's chains never saw. Detection and localisation both move with decoding, so section 3
records the source's real configuration rather than describing both as "the generator".

The generator is `Qwen2.5-3B-Instruct`, Act 2's shared instrument, so page 9 describes the same
population as the pages before it.

**On the PRM's lineage, stated accurately because an earlier draft of this notebook got it
wrong.** `Qwen2.5-Math-PRM-7B` is *not* from a different family. It is the same Qwen2.5 lineage
as the generator, on the math-specialised branch, so shared pretraining and shared error modes
are live possibilities rather than ruled out. What it is, is a different checkpoint with a
different specialisation and far more mathematical competence than the 3B it grades, which is
the right direction for a verifier: you do not want a grader as weak as the thing it grades.
Section 6 measures it instead of trusting it, and section 5 states the distribution shift that
measurement is looking for.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"       # "REPLAY" (no GPU), "QUICK" (small, generates), "FULL"
SEED = 123                # selects the items, and it is page 7's seed so the draws match

PRIMARY = "Qwen/Qwen2.5-3B-Instruct"      # pages 6 and 7's primary, so the pages compare
# Same Qwen2.5 lineage as the generator, math-specialised branch. NOT a different
# family, and section 5 says what that costs us.
PRM_MODEL = "Qwen/Qwen2.5-Math-PRM-7B"

PRESETS = {
    "REPLAY": dict(n_items=100, use_prm=False, may_generate=False),
    "QUICK":  dict(n_items=24,  use_prm=False, may_generate=True),
    "FULL":   dict(n_items=100, use_prm=True,  may_generate=True),
}
CFG = PRESETS[RUN_MODE]
USE_PRM = CFG["use_prm"]

# Page 6's top rung. A safety cap, not a budget: a chain that hits it is one WE cut,
# and section 3 drops those rather than grading them.
CHAIN_CAP = 2048

# Where a step ends. Passed at every call site because it is a claim, not a default
# (harness.step_spans has none). Same value as page 5, so a chain segmented here and
# a chain segmented there break the same way.
MIN_STEP_CHARS = 12

# ---- what the PRM has to clear before the page may quote it ----------------
# A probability calibrated on one distribution is not calibrated on another, so 0.5
# is a starting point rather than a threshold. Section 6 fits the operating point on
# the steps the checker labelled and records it as fitted.
PRM_THRESHOLD_START = 0.5
PRM_FIT_THRESHOLD = True

# Stated in advance so the gate is a pre-registration and not a post-hoc judgement.
# Specificity is the one that matters: a grader that flags a fifth of sound steps
# would make section 8's localisation meaningless, and marking down verbose prose it
# was not trained on is exactly how this PRM would fail on this generator's chains.
PRM_MIN_AUC = 0.65
PRM_MIN_SPECIFICITY = 0.80
PRM_MIN_COMPARED_STEPS = 30      # below this the reliability is too noisy to gate on

print(f"mode={RUN_MODE}  generator={PRIMARY}")
print(f"process graders: arithmetic checker" + (f" + {PRM_MODEL}" if USE_PRM else " only"))
if USE_PRM:
    print(f"the PRM publishes only if AUC >= {PRM_MIN_AUC:.2f} and specificity >= "
          f"{PRM_MIN_SPECIFICITY:.0%} over >= {PRM_MIN_COMPARED_STEPS} compared steps")

In [ ]:
import json
import random
import re
import textwrap
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test, render_prompt, extract_final_candidate, grade_answer)

OUT_DIR = Path("out/page_09")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

P7_DIR = Path("out/page_07")
P7_TRACES = P7_DIR / "traces.jsonl"
P7_MANIFEST = P7_DIR / "manifest.json"
TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"

mean, median = harness.mean, harness.median

print("torch", torch.__version__)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## 1. The outcome grader

Inherited unchanged from ch03, and it is the grader this whole page is arguing about.
`extract_final_candidate` pulls the last `\boxed{...}`; `grade_answer` checks equivalence.

Note what it can see: the last line. It has no opinion about the hundred tokens before it, which
is the entire subject of this page. The fifth check below is the page in miniature.


In [ ]:
CHECKS = [
    (r"The answer is \boxed{9}.",                        "9",    True),
    (r"27",                                              "9",    False),
    (r"\boxed{\frac{1}{2}}",                             "0.5",  True),
    (r"\boxed{1,024}",                                   "1024", True),
    # right answer via nonsense: the outcome grader says yes, which is the page's subject
    (r"Cancel the 6s: 16/64 -> 1/4. So \boxed{\frac{1}{4}}", "0.25", True),
    (r"I am not sure.",                                  "9",    False),
]

rows = [(raw, gt, exp, bool(grade_answer(extract_final_candidate(raw), gt)))
        for raw, gt, exp in CHECKS]
bad = [r for r in rows if r[3] != r[2]]
if bad:
    for raw, gt, exp, got in rows:
        print(f"{'ok ' if got == exp else 'FAIL'} {extract_final_candidate(raw)!r:<26} "
              f"vs {gt!r:<8} -> {got}")
    raise AssertionError(f"{len(bad)} outcome-grader check(s) behaved unexpectedly")

print(f"outcome grader preflight: {len(rows)}/{len(rows)} as expected")
print("the fifth case is not a bug in the grader. A right answer reached by nonsense is")
print("correct as far as an outcome check can see, and measuring how often that happens")
print("is what this page is for.")


## 2. The item set

MATH-500, levels 4 and 5, page 6's band, so this page and the ones before it describe the same
population.

Two reasons the band matters here specifically. **Wrong answers**: section 8 has nothing to
localise in without them. **Chain length**: a one-line solution cannot show a break partway
through, and easy problems get one-line solutions.


In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")

band = sorted((r for r in math500 if int(r["level"]) in (4, 5)),
              key=lambda r: r["unique_id"])          # sort before shuffling
random.Random(SEED).shuffle(band)
items = band[: CFG["n_items"]]
ITEMS_BY_ID = {r["unique_id"]: r for r in items}
ITEM_SET_HASH = harness.sha256_of(r["unique_id"] for r in items)

lv = "+".join(f"L{n}x{c}" for n, c in sorted(Counter(int(r["level"]) for r in items).items()))
print(f"{len(items)} items of {len(band)} available at levels 4-5   {lv}   "
      f"sha256 {ITEM_SET_HASH[:12]}")

# Page 7 asserts its draw against page 6's rather than trusting that two notebooks with
# the same constants drew the same items. Do the same here: a divergent draw would not
# fail, it would silently shrink the pool in section 3 and look like a thin run.
P7_MANIFEST_DATA = json.loads(P7_MANIFEST.read_text()) if P7_MANIFEST.exists() else None
SAME_ITEMS_AS_PAGE_7 = None
if P7_MANIFEST_DATA:
    p7_items = P7_MANIFEST_DATA.get("items", {})
    SAME_ITEMS_AS_PAGE_7 = p7_items.get("item_set_sha256") == ITEM_SET_HASH
    print(f"page 7 item set  {str(p7_items.get('item_set_sha256'))[:12]}  "
          f"n={p7_items.get('n')}  seed={p7_items.get('seed')}")
    print(f"-> {'identical draw' if SAME_ITEMS_AS_PAGE_7 else 'DIFFERENT draw from page 7'}")
    if not SAME_ITEMS_AS_PAGE_7:
        print("   Only the overlap is gradable, and the pages then describe different")
        print("   populations. Match page 7's seed and n_items, or say so on the page.")
else:
    print(f"{P7_MANIFEST} not found; the draw cannot be checked against page 7's")

## 3. Which chains this page is allowed to grade

The filter, and the one design decision here most worth understanding.

Grading a chain for bad reasoning only means something if the reasoning is the model's own. Three
sources, and only some of each qualifies:

| source | qualifies? |
|---|---|
| page 7's **pile** | yes, where `truncated` is false. These run to their natural end. |
| page 7's **split** axis | no. Each chain gets `B/N` tokens by construction. |
| page 6's sweep | no. Capped and force-committed on purpose; that is its experiment. |

So this is a filter on `truncated`, not a blanket refusal to reuse. If nothing survives it and
the mode may generate, the notebook runs its own uncapped pass instead.

**The filter is a selection, and the page has to say so rather than only the notebook.** A chain
hits the 2048-token cap because it was long, and it was long because the problem was hard, so
dropping cap-hitting chains preferentially drops hard problems and the failures attached to
them. Everything below is therefore conditional on *chains the model ended itself*, which is a
narrower population than *all chains* and skews easier. The cell measures the skew instead of
waving at it: it prints the level mix and the outcome accuracy of what survived against what was
dropped, and both land in `derived.json` so the caption can carry the real number.

**One chain per problem, and which one is arbitrary on purpose.** Page 7's pile is `POOL_N`
independent samples of the same problem, so taking the first by `sample_index` is one unbiased
draw. It is not the best chain, the modal chain, or the greedy chain, and it must not be
described as any of those: sections 7 and 8 describe what a single sample from this model looks
like, which is the thing an outcome grader would actually be handed.

In [ ]:
def usable(rec):
    """A chain whose ending was the model's choice, not ours."""
    return (not rec.get("truncated")) and not rec.get("forced") and rec.get("tokens")


def pick_pool_config(recs):
    """Name page 7's pile from its manifest, or recover it from the records.

    Page 7's dump holds several configurations and only ONE of them qualifies: the
    split axis is budget-starved by construction, the diversity family sits at
    temperatures the pile does not, and the temperature-0 configuration is one chain
    replicated. Pooling them would grade four decoding regimes at once and then call
    the mixture "the generator", which is the provenance defect this cell exists to
    avoid. Prefer the manifest; the fallback is the full-cap sampled configuration
    with the most records, and it says out loud that it guessed.
    """
    named = (P7_MANIFEST_DATA or {}).get("pile", {}).get("config_id")
    if named:
        return named, "page 7's manifest"
    at_cap = [r for r in recs
              if r.get("chain_budget") == CHAIN_CAP and (r.get("temperature") or 0) > 0]
    if not at_cap:
        return None, None
    guess = Counter(r["config_id"] for r in at_cap).most_common(1)[0][0]
    print(f"page 7's manifest is absent; inferring the pile as {guess!r} from the records")
    return guess, "inferred from the records"


source, records = None, []
CHAIN_PROVENANCE, FILTER_AUDIT = None, None

if P7_TRACES.exists():
    latest = {}
    for line in P7_TRACES.open(encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            latest[(r["run_id"], r.get("config_id"), r["problem_id"],
                    r.get("sample_index", 0))] = r
    raw7 = list(latest.values())
    P7_RUN_ID = Counter(r["run_id"] for r in raw7).most_common(1)[0][0]
    POOL_CONFIG, POOL_CONFIG_FROM = pick_pool_config(
        [r for r in raw7 if r["run_id"] == P7_RUN_ID])

    pool = [r for r in raw7
            if r["run_id"] == P7_RUN_ID
            and r.get("config_id") == POOL_CONFIG
            and r.get("model_id", PRIMARY) == PRIMARY
            and r["problem_id"] in ITEMS_BY_ID]
    kept = [r for r in pool if usable(r)]
    dropped = [r for r in pool if not usable(r)]

    # One chain per problem, the first by sample index, so no problem is counted twice.
    by_pid = {}
    for r in sorted(kept, key=lambda r: (r["problem_id"], r.get("sample_index", 0))):
        by_pid.setdefault(r["problem_id"], r)

    print(f"page 7 run {P7_RUN_ID}")
    print(f"  pile config {POOL_CONFIG!r} ({POOL_CONFIG_FROM})")
    print(f"  {len(pool):,} records on our items, {len(kept):,} ran to their natural end "
          f"({len(kept)/max(len(pool),1):.0%}),")
    print(f"  covering {len(by_pid)} of {len(items)} problems")

    if pool:
        # What the filter costs us, measured rather than asserted. If the dropped
        # chains are markedly harder or markedly more often wrong, sections 7 and 8
        # describe an easier population than "this model on MATH-500 L4-5".
        def _mix(rs):
            return "+".join(f"L{n}x{c}" for n, c in
                            sorted(Counter(int(r.get("level", 0)) for r in rs).items()))

        FILTER_AUDIT = {
            "n_records": len(pool), "n_kept": len(kept), "n_dropped": len(dropped),
            "kept_outcome_accuracy": mean([r["correct"] for r in kept]) if kept else None,
            "dropped_outcome_accuracy": (mean([r["correct"] for r in dropped])
                                         if dropped else None),
            "kept_level_mix": _mix(kept), "dropped_level_mix": _mix(dropped),
            "n_problems_covered": len(by_pid), "n_problems_asked": len(items),
        }
        harness.table(
            "what the natural-ending filter selected",
            ["chains", "n", "levels", "outcome accuracy"],
            [["the model ended them", len(kept), FILTER_AUDIT["kept_level_mix"],
              f"{FILTER_AUDIT['kept_outcome_accuracy']:.0%}" if kept else "n/a"],
             ["we cut them at the cap", len(dropped), FILTER_AUDIT["dropped_level_mix"],
              f"{FILTER_AUDIT['dropped_outcome_accuracy']:.0%}" if dropped else "n/a"]],
            note="everything below is conditional on the first row, which is the easier "
                 "of the two populations")

    if len(by_pid) >= max(8, CFG["n_items"] // 4):
        source = "page_07_pile"
        first = next(iter(by_pid.values()))
        CHAIN_PROVENANCE = {
            "source": source,
            "page_07_run_id": P7_RUN_ID,
            "page_07_config_id": POOL_CONFIG,
            "config_id_from": POOL_CONFIG_FROM,
            "decoding": "sampled",
            "temperature": first.get("temperature"),
            "top_p": first.get("top_p"),
            "chain_budget": first.get("chain_budget"),
            "prompt_template": (P7_MANIFEST_DATA or {}).get("prompt_template"),
            "prompt_suffix": None,     # page 7's chains never saw this page's suffix
            "chain_per_problem": "the first by sample_index, one unbiased draw of many",
        }
        print(f"\n  decoding: sampled at T={CHAIN_PROVENANCE['temperature']}, "
              f"top_p={CHAIN_PROVENANCE['top_p']}, cap {CHAIN_PROVENANCE['chain_budget']}")
        print("  These chains were NOT generated with this notebook's step-per-line")
        print("  suffix, and the manifest records that rather than implying otherwise.")
        records = [dict(r, question=ITEMS_BY_ID[r["problem_id"]]["problem"],
                        ground_truth=ITEMS_BY_ID[r["problem_id"]]["answer"],
                        outcome_correct=bool(r["correct"])) for r in by_pid.values()]
    else:
        print("        too few to stand on; generating instead")
else:
    print(f"{P7_TRACES} not found")

if not records and not CFG["may_generate"]:
    raise RuntimeError(
        f"REPLAY has no chains to grade: {P7_TRACES} is absent or nothing in it survived the "
        f"filter. Run page_07_frequency_voting.ipynb, or use QUICK to generate a small pass here.")

print(f"\nsource: {source or 'this notebook (see the next cell)'}")

In [ ]:
COT_SUFFIX = "\n\nSolve this step by step, one step per line."

if not records:
    device = get_device()
    dtype = harness.pick_dtype(device)
    loaded = harness.load_model(PRIMARY, device=device, dtype=dtype)
    model, tok = loaded["model"], loaded["tokenizer"]
    eos_ids = harness.end_of_turn_ids(tok)

    store = harness.TraceStore(TRACES_PATH, run_id=f"{RUN_MODE.lower()}-{ITEM_SET_HASH[:8]}",
                               key_fields=("problem_id",))
    store.report()

    for row in harness.progress(items, desc="generating", unit="item"):
        hit = store.get(row["unique_id"])
        if hit:
            records.append(hit)
            continue
        g = harness.generate(render_prompt(row["problem"]) + COT_SUFFIX, model, tok,
                             enable_thinking=False, max_new_tokens=CHAIN_CAP, eos_ids=eos_ids)
        extracted = extract_final_candidate(g["text"])
        records.append(store.append({
            "problem_id": row["unique_id"], "level": int(row["level"]),
            "subject": row["subject"], "question": row["problem"],
            "ground_truth": row["answer"], "trace": g["text"], "tokens": g["tokens"],
            "tokens_generated": g["tokens_generated"], "truncated": g["truncated"],
            "extracted_answer": extracted,
            "outcome_correct": bool(grade_answer(extracted, row["answer"])),
        }))

    cut = [r for r in records if r["truncated"]]
    if cut:
        print(f"\ndropping {len(cut)} chain(s) the cap cut: a chain WE ended is not a chain")
        print("the model got wrong, and grading it would report our cap as its error.")
        records = [r for r in records if not r["truncated"]]

    source = "generated_here"
    # A different decoding regime from page 7's pile and a different prompt, so it is
    # recorded as its own thing. A run that mixed the two sources and described them
    # with one line would be reporting a population that never existed.
    CHAIN_PROVENANCE = {
        "source": source,
        "decoding": "greedy",
        "temperature": 0.0, "top_p": None,
        "chain_budget": CHAIN_CAP,
        "prompt_template": render_prompt("{question}"),
        "prompt_suffix": COT_SUFFIX,
        "chain_per_problem": "the one chain generated per problem",
    }
    del model, tok, loaded            # the caller's references are what keep it alive
    harness.free_memory()

n_right = sum(r["outcome_correct"] for r in records)
print(f"\n{len(records)} chains: {n_right} right, {len(records) - n_right} wrong "
      f"({n_right/max(len(records),1):.0%} by the outcome grader)")
print(f"decoding: {CHAIN_PROVENANCE['decoding']}"
      + (f" at T={CHAIN_PROVENANCE['temperature']}"
         if CHAIN_PROVENANCE["decoding"] == "sampled" else ""))

## 4. The arithmetic checker

Segmentation comes from `harness.step_spans`, the same rule pages 5 and 7 use, so a chain broken
into steps here breaks the same way there. The checker itself stays in this notebook, because it
is a claim rather than plumbing and a reader has to be able to read it.

### The precision rule, which is the whole design

The checker may judge **only claims that are closed-form numeric on both sides, with no symbols.**
`3/4 = 9/12` is checkable. `Let x = 5` is a definition, not a false claim. `f(2) = 7` depends on a
definition made earlier in the chain. A checker that flagged those as false would be worse than no
checker at all, because the one thing this one has going for it is that its labels are facts a
reader can verify by hand.

Every step therefore gets one of three labels, and the third is not a hole to hide:

| label | meaning |
|---|---|
| `true` | every numeric claim in the step checks out |
| `false` | at least one numeric claim is arithmetically wrong |
| `unknown` | nothing in the step is a closed-form numeric claim |

High precision, low recall, stated openly. The preflight below includes the page's own canonical
example, cancelling the sixes in 16/64, precisely because the checker gets it **wrong in the
generous direction**: the arithmetic is true and the method is nonsense. That is the limit a
deterministic checker cannot cross, and it is why section 6 brings in a model.


In [ ]:
from sympy import simplify, sympify


def _to_sympy_text(s):
    """LaTeX-ish maths to something sympify can read. Deliberately narrow."""
    s = re.sub(r"\\left|\\right", "", s)
    s = re.sub(r"\\[dt]?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", s)
    s = (s.replace("\\times", "*").replace("\\cdot", "*").replace("\\div", "/")
          .replace("^", "**").replace("$", "").replace("\\", ""))
    return re.sub(r"\s+", "", s)


def numeric_value(text):
    """A sympy number if the text is closed-form numeric, else None.

    Any letter disqualifies it, which rejects 'x', 'f(2)', 'pi' and 'sqrt' alike. That
    costs recall and buys the only thing that matters here: no false positives."""
    t = _to_sympy_text(text)
    if not t or not re.search(r"\d", t) or re.search(r"[A-Za-z]", t):
        return None
    try:
        e = sympify(t, rational=True)
    except Exception:
        return None
    return e if getattr(e, "is_number", False) and not e.free_symbols else None


def _strip_prose(side):
    """Drop whole words of prose from the ends of one side of an equation.

    'So 8 * 7' -> '8 * 7' and '56 apples' -> '56': a claim does not stop being checkable
    for having a sentence wrapped round it. Three guards stop this inventing claims that
    were never made, and each one is a false positive that testing actually produced:

    * only tokens of TWO OR MORE letters go. Stripping the 'n' from 'n + 1' would leave
      '+ 1', and 'Let n = 4, so n + 1 = 5' would read as the false claim 1 = 5.
    * 'f(2)' is one token and is not purely alphabetic, so it survives, meets the
      no-letters rule and is skipped, rather than being mined for a '(2)'.
    * if stripping leaves the side starting on an operator we amputated an operand, so
      refuse to judge it at all.
    """
    toks = side.split()
    stripped = False
    while toks and (toks[0].endswith(":")                      # a step label, 'Step 3:'
                    or (toks[0].isalpha() and len(toks[0]) > 1)):
        toks.pop(0)
        stripped = True
    while toks and toks[-1].isalpha() and len(toks[-1]) > 1:
        toks.pop()
    out = " ".join(toks).rstrip(".")
    return "" if stripped and re.match(r"^[+*/^]|^-\s", out) else out


def check_step(text):
    """-> (label, judged_claim). A step is false if ANY numeric claim in it is."""
    # Thousands separators first: splitting on commas below would cut '1,024' in half
    # and turn a valid claim into an unparseable one.
    text = re.sub(r"(?<=\d),(?=\d\d\d\b)", "", text)

    judged = None
    for chunk in re.split(r"[,;]|\bso\b|\bSo\b|\bthus\b|\band\b|\bTherefore\b", text):
        if "=" not in chunk:
            continue
        vals = [numeric_value(_strip_prose(p)) for p in chunk.split("=")]
        if len(vals) < 2 or any(v is None for v in vals):
            continue
        judged = chunk.strip()
        if any(simplify(vals[0] - v) != 0 for v in vals[1:]):
            return "false", judged
    return ("true", judged) if judged else ("unknown", None)


In [ ]:
PREFLIGHT = [
    # --- claims it should judge ------------------------------------------------
    ("3/4 = 9/12",                       "true"),
    ("17/12 - 5/12 = 1",                 "true"),
    ("2 + 2 = 5",                        "false"),
    ("So 8 * 7 = 54.",                   "false"),   # prose in front of the claim
    ("Step 3: 100 - 37 = 62",            "false"),   # a step label in front of it
    ("We get 8 * 7 = 56 apples",         "true"),    # units after it
    ("1,024 = 1024",                     "true"),    # a separator, not a chunk break
    (r"\frac{3}{4} = \frac{9}{12}",      "true"),
    (r"$5 \times 4 = 20$",               "true"),
    ("The total is -5 = -5",             "true"),
    # --- claims it must refuse: each is a false positive waiting to happen ------
    ("Let x = 5",                        "unknown"),  # a definition, not a claim
    ("f(2) = 7",                         "unknown"),  # depends on an earlier definition
    ("Let n = 4, so n + 1 = 5",          "unknown"),  # dropping 'n' would leave '+1 = 5'
    ("Substituting, 2x = 10",            "unknown"),  # an unknown in it
    ("x = 3 and y = 4",                  "unknown"),
    (r"\sqrt{16} = 4",                   "unknown"),  # true, but not closed-form here
    ("(3, 5) = (3, 5)",                  "unknown"),  # a tuple, not arithmetic
    ("The common denominator is 12",     "unknown"),  # no claim at all
    ("The area is 12 square units",      "unknown"),
    # --- the blind spot, as a test rather than as a caveat ---------------------
    ("Cancel the 6s, so 16/64 = 1/4",    "true"),     # arithmetic true, method nonsense
]

fails = 0
for text, expect in PREFLIGHT:
    got, claim = check_step(text)
    if got != expect:
        fails += 1
        print(f"FAIL {text!r:<38} -> {got:<8} expected {expect}")
assert not fails, f"{fails} of {len(PREFLIGHT)} checker preflight cases behaved unexpectedly"

print(f"checker preflight: {len(PREFLIGHT)}/{len(PREFLIGHT)} as expected")
print(f"  {sum(1 for _, e in PREFLIGHT if e != 'unknown')} it judges, "
      f"{sum(1 for _, e in PREFLIGHT if e == 'unknown')} it refuses")
print("\nThe last case is the one to notice: 'cancel the 6s, so 16/64 = 1/4' comes back TRUE.")
print("The arithmetic is right and the method is nonsense, and no deterministic checker can")
print("tell those apart. That gap is what section 6 is for.")


In [ ]:
def delimiter_of(text):
    """Which rule in `harness.step_spans`' cascade actually divided this chain.

    Recorded because it decides whether the PRM in section 5 is being asked to score
    at positions its own convention would produce. That checkpoint's released usage
    splits a response on BLANK LINES and marks each step end, so a chain we cut on
    single newlines or on sentence boundaries is scored somewhere else entirely. This
    mirrors the cascade rather than reaching into it: same patterns, same order, and
    it only asks which one first yields more than one piece.
    """
    text = harness.strip_empty_think(text)
    for name, pattern in (("paragraph", r"\n\s*\n"), ("line", r"\n"),
                          ("sentence", r"(?<=[.!?])\s+")):
        if len([p for p in re.split(pattern, text) if p.strip()]) > 1:
            return name
    return "whole chain"


for r in harness.progress(records, desc="checking steps", unit="chain"):
    steps = harness.segment_steps(r["trace"], r["tokens"], min_step_chars=MIN_STEP_CHARS)
    for s in steps:
        s["check"], s["claim"] = check_step(s["text"])
    r["steps"] = steps
    r["n_steps"] = len(steps)
    r["delimiter"] = delimiter_of(r["trace"])
    r["first_false"] = next((i for i, s in enumerate(steps) if s["check"] == "false"), None)
    r["n_false"] = sum(s["check"] == "false" for s in steps)
    r["n_checkable"] = sum(s["check"] in ("true", "false") for s in steps)

all_steps = [s for r in records for s in r["steps"]]
lab = Counter(s["check"] for s in all_steps)

harness.table(
    "step labels",
    ["label", "steps", "share"],
    [[k, lab[k], f"{lab[k]/max(len(all_steps),1):.1%}"] for k in ("true", "false", "unknown")],
    note=f"{len(all_steps):,} steps over {len(records)} chains; "
         f"median {median([r['n_steps'] for r in records]):.0f} steps per chain")

DELIMITER_MIX = Counter(r["delimiter"] for r in records)
harness.table(
    "what divided each chain into steps",
    ["rule", "chains", "share"],
    [[k, v, f"{v/max(len(records),1):.0%}"] for k, v in DELIMITER_MIX.most_common()],
    note="'paragraph' matches the PRM's own convention; anything else scores it "
         "off-convention and section 6 has to carry that")

print(f"\nchains with at least one step the checker could judge: "
      f"{sum(r['n_checkable'] > 0 for r in records)}/{len(records)}")
if lab["unknown"] / max(len(all_steps), 1) > 0.8:
    print("Most steps are beyond a deterministic checker, which is itself the argument for")
    print("section 6 and worth saying on the page rather than hiding.")
if DELIMITER_MIX["paragraph"] / max(len(records), 1) < 0.5 and USE_PRM:
    print("\nUnder half of these chains split on blank lines, so most of them are being")
    print("cut where the PRM's convention would not cut them. Expect its scores to be")
    print("worse than the checkpoint's reported quality, and read section 6 as measuring")
    print("THIS pairing rather than the PRM in general.")

## 5. The process reward model

The checker cannot see method errors. A model can, and `Qwen2.5-Math-PRM-7B` emits a score per
reasoning step, which is process supervision in the literal sense.

But a PRM's labels are opinions, and putting them on the page unexamined would repeat exactly the
defect this refactor exists to remove: the current page reveals hand-written booleans and calls
them a PRM's verdict. So the PRM does not get to be the authority here. It gets **measured**, in
section 6, against the checker on the steps both can see, because on its own subset the checker
is right by construction.

### What could be wrong with these labels, named before they are collected

**Not its competence.** The PRM is far better at mathematics than the 3B whose work it grades,
and that is the right direction for a verifier rather than a problem with one. A grader no
stronger than the thing it grades is the useless case.

**Its lineage, which is not independent.** It is the same Qwen2.5 pretraining lineage as the
generator, on the math-specialised branch. So it is a test of whether a *bigger, specialised*
checkpoint catches this one's errors, not of whether an *unrelated* grader does, and any error
mode the two inherited together is invisible to this pairing. Page 8's notebook carries the same
limitation for its 7B judge and says so; this page says so too.

**Distribution shift, which is the one to actually watch.** Its step labels were learned on
terse, LaTeX-dense rollouts from math-specialised models with steps delimited by blank lines.
This page feeds it a chatty 3B-Instruct chain segmented by whatever rule section 4 printed. The
failure that produces is not random noise, it is **low specificity**: sound-but-verbose steps
marked down for being unlike the training distribution. That would inflate the PRM's detection
row in section 7 and would make localisation meaningless if localisation depended on it, which
is why sections 7 and 8 are the checker's and the PRM only ever adds a second row.

So section 6 measures agreement, AUC and specificity, and section 2's gate decides whether the
PRM may appear on the page at all. If it is off, will not load, or fails either preflight,
everything below still runs from the checker alone and `derived.json` records that it did not
participate.

**A bug in the next cell is not a missing checkpoint.** An earlier draft caught every exception
around the load, so a `NameError` in this notebook printed "PRM unavailable", fell through to
checker-only, and published a full set of numbers under a `FULL` label. Programming errors are
re-raised here; only genuine load failures fall back.

In [ ]:
PRM_READY, prm, prm_tok = False, None, None
PRM_PREFLIGHT = None
STEP_SEP = "<extra_0>"          # the token this PRM scores at, one per step

if USE_PRM:
    from transformers import AutoModel, AutoTokenizer
    try:
        device = get_device()
        dtype = harness.pick_dtype(device)
        print(f"[1/2] fetch      {PRM_MODEL}")
        prm_path = harness.fetch(PRM_MODEL)
        print("[2/2] load       PRM weights")
        prm_tok = AutoTokenizer.from_pretrained(prm_path, trust_remote_code=True)
        prm = AutoModel.from_pretrained(prm_path, dtype=dtype,
                                        trust_remote_code=True).to(device).eval()
        got_dtype = next(prm.parameters()).dtype
        if got_dtype != dtype:
            raise RuntimeError(
                f"asked for {dtype} and the PRM loaded as {got_dtype}; the manifest would "
                f"assert a precision this run never used")
        PRM_READY = True
    except (NameError, AttributeError, TypeError, SyntaxError, IndentationError):
        # A bug in THIS cell, not a missing checkpoint. Falling back here would publish
        # a checker-only run under a FULL label, which is the one failure this notebook
        # must not have. Let it break loudly.
        raise
    except Exception as e:
        print(f"\nPRM unavailable ({type(e).__name__}: {e})")
        print("Continuing with the checker alone; sections 7 and 8 still produce their numbers.")
else:
    print(f"USE_PRM is off in {RUN_MODE}. The checker carries every measurement below.")


@torch.inference_mode()
def prm_scores(question, step_texts):
    """One score in [0,1] per step. Higher means the PRM thinks the step is sound."""
    body = STEP_SEP.join(t.strip() for t in step_texts) + STEP_SEP
    chat = prm_tok.apply_chat_template(
        [{"role": "system", "content": "Please reason step by step."},
         {"role": "user", "content": question},
         {"role": "assistant", "content": body}],
        tokenize=False, add_generation_prompt=False)
    enc = prm_tok(chat, return_tensors="pt").to(prm.device)
    out = prm(**enc)
    logits = out[0] if isinstance(out, (tuple, list)) else getattr(out, "logits", out)
    sep_id = prm_tok.encode(STEP_SEP, add_special_tokens=False)[0]
    probs = torch.softmax(logits.float(), dim=-1)[enc["input_ids"] == sep_id]
    return probs[:, -1].tolist()


if PRM_READY:
    # A clean chain against the same chain with one step corrupted. If the PRM cannot
    # separate those, its scores are noise and nothing below should use them.
    q = "What is 3/4 + 2/3?"
    clean = ["The common denominator is 12.", "3/4 = 9/12 and 2/3 = 8/12.",
             "9/12 + 8/12 = 17/12.", "So the answer is \\boxed{17/12}."]
    dirty = list(clean)
    dirty[2] = "9/12 + 8/12 = 15/12."                    # now arithmetically false

    a, b = prm_scores(q, clean), prm_scores(q, dirty)
    assert len(a) == len(clean), (
        f"PRM returned {len(a)} scores for {len(clean)} steps; the separator convention "
        f"for this checkpoint is not what this cell assumes")

    harness.table("PRM separation preflight", ["step", "clean", "corrupted"],
                  [[i + 1, f"{x:.3f}", f"{y:.3f}" + ("   <- corrupted" if i == 2 else "")]
                   for i, (x, y) in enumerate(zip(a, b))])

    PRM_PREFLIGHT = {"clean": a, "corrupted": b, "margin": a[2] - b[2],
                     "separates": bool(b[2] < a[2])}
    print(f"\nseparation margin on the corrupted step: {PRM_PREFLIGHT['margin']:+.3f}")
    print("Note what this preflight is and is not. These four steps are terse and")
    print("blank-line clean, which is the PRM's home distribution, so passing here says")
    print("the plumbing works. It does not say the scores transfer to the chains in")
    print("section 4, and that is what section 6 is for.")

    if not PRM_PREFLIGHT["separates"]:
        PRM_READY = False
        print("\nIt does not mark a known-false step lower than a true one, so its scores are")
        print("not usable as labels. Falling back to the checker alone.")

In [ ]:
PRM_FAILURES = Counter()

if PRM_READY:
    for r in harness.progress(records, desc="PRM scoring", unit="chain"):
        try:
            sc = prm_scores(r["question"], [s["text"] for s in r["steps"]])
        except (NameError, AttributeError, TypeError):
            raise                       # our bug, not this chain's fault
        except Exception as e:
            PRM_FAILURES[type(e).__name__] += 1
            sc = []
        r["prm_ok"] = len(sc) == len(r["steps"])
        if not r["prm_ok"]:
            if sc:
                PRM_FAILURES["score_count_mismatch"] += 1
            continue
        for s, v in zip(r["steps"], sc):
            s["prm"] = float(v)         # raw score only; the threshold is fitted below
    n_ok = sum(r.get("prm_ok", False) for r in records)
    print(f"\nPRM scored {n_ok}/{len(records)} chains")
    if PRM_FAILURES:
        print(f"  chains it could not score: {dict(PRM_FAILURES)}")
    print("  Scores are stored raw. Nothing is called sound or unsound yet, because the")
    print("  cut-off is a fitted quantity and section 6 is where it gets fitted.")
else:
    for r in records:
        r["prm_ok"] = False


def apply_prm_labels(threshold):
    """Turn raw PRM scores into step labels at one cut-off, and roll them up per chain.

    Kept separate from scoring so the threshold can move without a second forward
    pass, and so that a reader can see that the labels are a function of a number this
    notebook chose rather than a verdict the model handed down.
    """
    for r in records:
        if not r.get("prm_ok"):
            continue
        for s in r["steps"]:
            s["prm_label"] = "false" if s["prm"] < threshold else "true"
        r["prm_first_false"] = next(
            (i for i, s in enumerate(r["steps"]) if s["prm_label"] == "false"), None)
        r["prm_n_false"] = sum(s["prm_label"] == "false" for s in r["steps"])

## 6. Is the PRM worth believing?

On steps the checker judged, how well do the PRM's scores separate true from false? Where the
two disagree the checker is right, by construction, so this is a measurement of the PRM and not
a negotiation between two opinions.

**AUC is the headline, and the threshold is fitted rather than assumed.** An earlier draft
called a step unsound below 0.5 and then reported accuracy at that cut-off. A probability
calibrated on one distribution is not calibrated on another, so 0.5 was measuring where this
checkpoint's scores happen to sit on our chains as much as it was measuring the checkpoint. AUC
asks the threshold-free question, *does it rank a false step below a true one*, which is what a
process grader is actually for and is what page 8's spec demands of every verifier. The
operating point is then chosen on the same labelled steps and **recorded as fitted**, because a
number we picked from the data is not a property of the model.

Three readings at that operating point, answering different questions. **Agreement** is the
summary. **Recall on false steps** is whether it catches breaks the checker proved were there.
**Specificity** is whether it cries wolf on steps that were fine, and it is the one that decides
publication: a grader that flags a fifth of sound steps would drown section 8's localisation in
false breaks, and marking down verbose prose it was never trained on is precisely how this
pairing fails.

**The deepest caveat, and it belongs on the page rather than only here.** The checker judges
only closed-form arithmetic. Those steps are terse and numeric, which is to say they are the
steps *most* like the PRM's training distribution. So everything measured here is an
**optimistic** estimate of how the PRM does on the prose steps where it is actually being
trusted, and section 4 has probably already reported that those prose steps are most of the
chain. The reliability is measured where the checker could reach, and that is the easy ground
for both graders.

Whatever comes out is attached to any PRM-only claim the page makes. If the gate in section 2
fails, the PRM's rows are suppressed rather than footnoted.

In [ ]:
RELIABILITY, PRM_PUBLISHABLE = None, False
PRM_THRESHOLD = PRM_THRESHOLD_START

pairs = [(s["check"], s["prm"]) for r in records if r.get("prm_ok")
         for s in r["steps"] if s["check"] in ("true", "false") and "prm" in s]


def auc_of(ps):
    """P(a true step outranks a false one), ties at half credit.

    The Mann-Whitney reading of AUC, written out rather than imported because it is
    three lines and the whole point of this section is that a reader can check it.
    None when the checker labelled only one class, which is not a score of 0.5: it is
    no measurement at all, and the gate below treats it that way.
    """
    t = [v for lab, v in ps if lab == "true"]
    f = [v for lab, v in ps if lab == "false"]
    if not t or not f:
        return None
    wins = sum((x > y) + 0.5 * (x == y) for x in t for y in f)
    return wins / (len(t) * len(f))


def readings_at(ps, threshold):
    """Agreement, recall on false steps and specificity on true ones, at one cut-off."""
    got = [(lab, "false" if v < threshold else "true") for lab, v in ps]
    on_false = [g for lab, g in got if lab == "false"]
    on_true = [g for lab, g in got if lab == "true"]
    return {
        "threshold": threshold,
        "agreement": mean([lab == g for lab, g in got]),
        "recall_on_false": mean([g == "false" for g in on_false]) if on_false else None,
        "specificity_on_true": mean([g == "true" for g in on_true]) if on_true else None,
    }


if pairs:
    AUC = auc_of(pairs)
    at_start = readings_at(pairs, PRM_THRESHOLD_START)

    if PRM_FIT_THRESHOLD and AUC is not None:
        # Youden's J: the cut-off that best trades catching false steps against not
        # flagging true ones. Fitted on the checker's labels, which is the only ground
        # truth here, and recorded as fitted so nobody reads it as the model's own.
        vals = sorted({v for _, v in pairs})
        if len(vals) > 200:                       # keep the search cheap on a big pile
            vals = [vals[int(i * len(vals) / 200)] for i in range(200)]
        cands = [0.0] + [(a + b) / 2 for a, b in zip(vals, vals[1:])] + [1.0]

        def youden(t):
            rd = readings_at(pairs, t)
            if rd["recall_on_false"] is None or rd["specificity_on_true"] is None:
                return -1.0
            return rd["recall_on_false"] + rd["specificity_on_true"] - 1.0

        PRM_THRESHOLD = max(cands, key=youden)

    fitted = readings_at(pairs, PRM_THRESHOLD)
    apply_prm_labels(PRM_THRESHOLD)

    PRM_PUBLISHABLE = bool(
        len(pairs) >= PRM_MIN_COMPARED_STEPS
        and AUC is not None and AUC >= PRM_MIN_AUC
        and fitted["specificity_on_true"] is not None
        and fitted["specificity_on_true"] >= PRM_MIN_SPECIFICITY)

    n_false_steps = sum(1 for lab, _ in pairs if lab == "false")
    RELIABILITY = {
        "n_steps_compared": len(pairs),
        "n_false_steps": n_false_steps,
        "auc": AUC,
        "threshold": PRM_THRESHOLD,
        "threshold_source": "fitted on the checker's labels (Youden's J)"
                            if PRM_FIT_THRESHOLD else "fixed",
        "threshold_start": PRM_THRESHOLD_START,
        "agreement": fitted["agreement"],
        "recall_on_false": fitted["recall_on_false"],
        "specificity_on_true": fitted["specificity_on_true"],
        "at_start_threshold": at_start,
        "gate": {"min_auc": PRM_MIN_AUC, "min_specificity": PRM_MIN_SPECIFICITY,
                 "min_compared_steps": PRM_MIN_COMPARED_STEPS, "passed": PRM_PUBLISHABLE},
        "measured_on": "steps with a closed-form numeric claim, which are the steps most "
                       "like the PRM's training distribution; an optimistic estimate of "
                       "its reliability on the prose steps it is trusted with",
    }

    def _pct(x):
        return f"{x:.1%}" if x is not None else "n/a"

    harness.table(
        "PRM against the arithmetic checker",
        ["measure", "value", "over"],
        [["AUC (threshold-free)", f"{AUC:.3f}" if AUC is not None else "n/a",
          f"{len(pairs)} steps"],
         ["operating point", f"{PRM_THRESHOLD:.3f}", RELIABILITY["threshold_source"]],
         ["agreement", _pct(fitted["agreement"]), f"{len(pairs)} steps"],
         ["recall on false steps", _pct(fitted["recall_on_false"]),
          f"{n_false_steps} steps"],
         ["specificity on true steps", _pct(fitted["specificity_on_true"]),
          f"{len(pairs) - n_false_steps} steps"]],
        note="where they disagree the checker is right, by construction")

    print(f"\nat the unfitted 0.5 cut-off, for contrast: agreement "
          f"{_pct(at_start['agreement'])}, recall {_pct(at_start['recall_on_false'])}, "
          f"specificity {_pct(at_start['specificity_on_true'])}")

    if PRM_PUBLISHABLE:
        print(f"\nGATE PASSED. The PRM's rows may appear, carrying AUC {AUC:.2f} and the")
        print("caveat that this was measured on the arithmetic steps rather than the prose.")
    else:
        why = []
        if len(pairs) < PRM_MIN_COMPARED_STEPS:
            why.append(f"only {len(pairs)} comparable steps (need {PRM_MIN_COMPARED_STEPS})")
        if AUC is None:
            why.append("AUC undefined: the checker labelled only one class of step, so "
                       "there is nothing to rank against")
        elif AUC < PRM_MIN_AUC:
            why.append(f"AUC {AUC:.2f} below {PRM_MIN_AUC:.2f}")
        if (fitted["specificity_on_true"] is not None
                and fitted["specificity_on_true"] < PRM_MIN_SPECIFICITY):
            why.append(f"specificity {_pct(fitted['specificity_on_true'])} below "
                       f"{PRM_MIN_SPECIFICITY:.0%}")
        print(f"\nGATE FAILED: {'; '.join(why)}.")
        print("The PRM's rows are suppressed below. This is a finding about grading this")
        print("generator's chains with this checkpoint, and it is worth one sentence on the")
        print("page: an off-the-shelf PRM pointed at off-distribution prose is not free.")
elif PRM_READY:
    print("no overlapping steps to compare on, so the PRM's reliability is unmeasured and")
    print("the page may not report PRM-only judgements")
else:
    print("no PRM in this run; every measurement below is the checker's")

## 7. Detection: how often does outcome grading reward nonsense

Spec §3a. Among chains that reached the **right** answer, the share containing a step the checker
proved false. Outcome grading passes every one.

**This may come out small, and the spec says so in advance.** A false step usually propagates
into a wrong answer, and the cases where it does not are often method errors with valid
arithmetic, which the checker cannot see. If it is small, the page reports it as small and
changes its verb rather than inflating it.

Small still matters, because of where it lands. Page 12 is STaR: generate rationales, keep the
ones that reach the right answer, fine-tune on those. This share is the contamination rate of
that filter.

**What population the rate is over, which the caption has to say.** One sample per problem, from
chains the model ended on its own, at the decoding section 3 named. Not the best of several, not
the greedy chain, and not the chains our cap cut off. Section 3 printed what that filter selected
away; if it selected away much, this rate reads low for a reason that is ours rather than the
model's.

In [ ]:
right = [r for r in records if r["outcome_correct"]]
wrong = [r for r in records if not r["outcome_correct"]]
right_bad = [r for r in right if r["n_false"] > 0]

DETECTION = {
    "grader": "arithmetic checker", "source": source,
    "n_right": len(right), "n_right_with_false_step": len(right_bad),
    "base_rate": len(right_bad) / max(len(right), 1),
    "n_wrong": len(wrong),
    "prm_reported": False,
}

rows = [["outcome grader", f"{len(right)} of {len(records)} chains pass", ""],
        ["of those, a step the checker proved false", len(right_bad),
         f"{DETECTION['base_rate']:.1%}"]]

if PRM_PUBLISHABLE:
    pr = [r for r in right if r.get("prm_ok")]
    pr_bad = [r for r in pr if r.get("prm_n_false", 0) > 0]
    DETECTION["prm_reported"] = True
    DETECTION["prm_base_rate"] = len(pr_bad) / max(len(pr), 1)
    DETECTION["prm_n_chains"] = len(pr)
    DETECTION["prm_auc"] = RELIABILITY["auc"]
    DETECTION["prm_threshold"] = RELIABILITY["threshold"]
    rows.append(["of those, a step the PRM calls unsound", len(pr_bad),
                 f"{DETECTION['prm_base_rate']:.1%}"])
    note = (f"the PRM row also sees non-arithmetic steps; attach its measured "
            f"AUC {RELIABILITY['auc']:.2f} to it, and the fact that the AUC was "
            f"measured on arithmetic steps only")
elif RELIABILITY:
    note = (f"the PRM ran and did not clear section 2's gate, so its row is suppressed "
            f"rather than footnoted; the checker sees arithmetic only, so this is a "
            f"floor rather than a rate")
    print("The PRM scored these chains and its row is withheld. The suppression is the")
    print("finding, not a gap: derived.json carries what it scored and why it failed.\n")
else:
    note = "the checker sees arithmetic only, so this is a floor rather than a rate"

harness.table("detection", ["", "count", "share of right answers"], rows, note=note)

if DETECTION["base_rate"] == 0:
    print("\nZERO -- a fact about the checker before it is a fact about the model. See the")
    print("spec's open question 4 before reporting this as a finding about reasoning.")
elif DETECTION["base_rate"] < 0.05:
    print("\nSmall. Report it as small; the STaR framing is what gives it stakes.")

## 8. Localisation: how much of a wrong chain was sound

Spec §3b, and the measurement Solution B on the current page has always taught without evidence.

Take chains that reached the **wrong** answer where the checker finds a break at step *k* of *n*.
Outcome grading can only say *wrong*, which condemns all *n*. Process grading points at *k* and
leaves the first *k-1* standing. So the quantity is `k/n`, the share of the chain that survives.

The second reading tests something the page asserts rather than shows: whether a break really is
one step, or whether real chains carry several.


In [ ]:
localisable = [r for r in wrong if r["first_false"] is not None and r["n_steps"] > 1]
LOCALISATION = {"n_wrong": len(wrong), "n_localisable": len(localisable)}

if localisable:
    shares = sorted(r["first_false"] / r["n_steps"] for r in localisable)
    med = shares[len(shares) // 2]
    dist = Counter(r["n_false"] for r in localisable)
    LOCALISATION.update({
        "median_share_sound": med,
        "mean_share_sound": mean(shares),
        "share_with_one_false_step": dist[1] / len(localisable),
        "n_false_distribution": dict(sorted(dist.items())),
    })

    harness.table(
        "what each grader condemns, on a wrong chain with a findable break",
        ["grader", "verdict", "share of the chain condemned"],
        [["outcome", "wrong", "100%"],
         ["process", f"broken at step {med:.0%} of the way in", f"{1 - med:.0%}"]],
        note=f"{len(localisable)} of {len(wrong)} wrong chains have a break the checker can find")

    print(f"\n-> the first {med:.0%} of the chain was sound, and outcome grading threw it away")
    print(f"\nbreaks per chain: {LOCALISATION['n_false_distribution']}")
    print(f"exactly one broken step: {LOCALISATION['share_with_one_false_step']:.0%}")
    if LOCALISATION["share_with_one_false_step"] < 0.5:
        print("  ^ under half, so the page must NOT claim breaks localise to a single line.")
        print("    Follow the data: say how many, and keep Solution B as one case, not the rule.")
else:
    print(f"No wrong chain has a findable break, out of {len(wrong)} wrong chains.")
    print("Section 8 has nothing to plot. Either the checker's recall is too low on this")
    print("model's output, or these answers fail for non-arithmetic reasons.")


In [ ]:
import matplotlib.pyplot as plt

if localisable:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2),
                                   gridspec_kw={"width_ratios": [3, 2]})
    ax1.hist([s * 100 for s in shares], bins=10, range=(0, 100), color="#3b4a6b")
    ax1.axvline(med * 100, color="#c2703d", lw=2)
    ax1.set_xlabel("share of the chain before the break (%)")
    ax1.set_ylabel("wrong chains")
    ax1.set_title("where the break lands", fontsize=10, loc="left")

    ax2.barh([1, 0], [100, (1 - med) * 100], height=0.5, color=["#8c8c8c", "#3b4a6b"])
    ax2.set_yticks([1, 0]); ax2.set_yticklabels(["outcome", "process"])
    ax2.set_xlim(0, 100); ax2.set_xlabel("% of the chain condemned")
    ax2.set_title("what each grader throws away", fontsize=10, loc="left")

    for ax in (ax1, ax2):
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(f"{PRIMARY} · MATH-500 L4-5 · n={len(localisable)} wrong chains with a "
                 f"findable break", fontsize=9, x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "figure_localisation.png", dpi=160, bbox_inches="tight")
    plt.show()


## 8b. Do the numbers survive a different idea of where a step ends?

Sections 7 and 8 rest on a segmentation rule, and a rule is a choice. `harness.step_spans` runs a
cascade, blank lines then single lines then sentences, and takes the first that divides the
chain, so different chains in this run were cut by different rules. Section 4 printed which.

This is not a footnote-sized worry. **A step is the unit both graders judge**, so cutting finer
produces more steps and more chances for the checker to find a false one, while cutting coarser
buries a false claim inside a step that also contains true ones. If the headline moves a lot
when the rule changes, then it is partly a measurement of our formatting convention and the page
has to say so.

Cheap to answer: force each rule on every chain and recompute. No inference, no model, no
generation, and it runs in `REPLAY`. The canonical row is what sections 7 and 8 reported; the
other three are what the same chains would have said under one rule throughout.

It also settles the PRM question from section 5 in the honest direction. The `paragraph` row is
the PRM's own convention, so the distance between it and the canonical row is roughly what the
PRM is being asked to absorb before it says anything at all.

In [ ]:
def resegment(text, pattern, min_step_chars=MIN_STEP_CHARS):
    """Step texts under one forced delimiter, with the cascade's merge rule for short
    fragments. Text only: the checker needs no token spans, and section 10's reveal
    keeps the canonical segmentation. This exists to answer one question and no other.
    """
    text = harness.strip_empty_think(text)
    pieces = [p.strip() for p in re.split(pattern, text) if p.strip()]
    merged = []
    for k, p in enumerate(pieces):
        short = (len(p) < min_step_chars and not re.search(r"\d", p)
                 and k != len(pieces) - 1 and "\\boxed" not in p)
        if merged and short:
            merged[-1] = merged[-1] + " " + p
        else:
            merged.append(p)
    return merged


def measure_under(step_lists):
    """Detection and localisation from one segmentation of every chain."""
    rows = []
    for r, steps in zip(records, step_lists):
        labels = [check_step(t)[0] for t in steps]
        rows.append(dict(
            correct=r["outcome_correct"], n=len(steps),
            n_false=sum(l == "false" for l in labels),
            first_false=next((i for i, l in enumerate(labels) if l == "false"), None)))
    rt = [x for x in rows if x["correct"]]
    wr = [x for x in rows if not x["correct"]]
    loc = [x["first_false"] / x["n"] for x in wr
           if x["first_false"] is not None and x["n"] > 1]
    return dict(mean_steps=mean([x["n"] for x in rows]) if rows else 0.0,
                detection=len([x for x in rt if x["n_false"] > 0]) / max(len(rt), 1),
                n_localisable=len(loc),
                median_share_sound=median(loc) if loc else None)


SEGMENTATION_CHECK = {"canonical": measure_under([[s["text"] for s in r["steps"]]
                                                  for r in records])}
for name, pattern in (("paragraph", r"\n\s*\n"), ("line", r"\n"),
                      ("sentence", r"(?<=[.!?])\s+")):
    SEGMENTATION_CHECK[name] = measure_under(
        [resegment(r["trace"], pattern) for r in records])

harness.table(
    "the same chains, cut four ways",
    ["segmentation", "steps/chain", "detection", "wrong chains with a break", "share sound"],
    [[k, f"{v['mean_steps']:.1f}", f"{v['detection']:.1%}", v["n_localisable"],
      f"{v['median_share_sound']:.0%}" if v["median_share_sound"] is not None else "n/a"]
     for k, v in SEGMENTATION_CHECK.items()],
    note="'canonical' is the cascade sections 7 and 8 actually used; the rest force one "
         "rule on every chain")

spread = [v["detection"] for v in SEGMENTATION_CHECK.values()]
SEGMENTATION_CHECK["detection_spread_pts"] = (max(spread) - min(spread)) * 100
print(f"\ndetection ranges over {SEGMENTATION_CHECK['detection_spread_pts']:.1f} points "
      f"across the four rules")
if SEGMENTATION_CHECK["detection_spread_pts"] > 5:
    print("That is large enough that the headline is partly a property of where we chose")
    print("to cut, not only of the reasoning. Report the canonical number WITH this range")
    print("beside it; a single figure would be overclaiming.")
else:
    print("Small, so the headline survives the choice of delimiter and the page can quote")
    print("the canonical number plainly. This is the check that earns it that right.")

## 9. Selection: a process-aware rung on page 8's ladder

Spec §3c. Page 8 ranks selectors that all read the final answer; this adds one that reads the
working. Among a problem's candidates, drop any containing a checker-false step, then take the
majority of what survives.

**Optional and self-skipping.** It needs page 7's candidate pile. If that is absent this reports
so and the page ships sections 7 and 8 alone, which are what it leads with anyway.

Expect a small gain, and report whatever comes out. A null result is a real finding about how
much of the failure is method rather than arithmetic, and it is one page 8's ladder is already
shaped to show.


In [ ]:
SELECTION = None

if not P7_TRACES.exists():
    print(f"{P7_TRACES} not found -- skipping the selector rung. Sections 7 and 8 stand alone.")
elif not POOL_CONFIG:
    print("page 7's pile could not be named, so there is no candidate set to select from.")
else:
    from collections import defaultdict
    cands = defaultdict(list)
    seen = {}
    for line in P7_TRACES.open(encoding="utf-8"):
        if line.strip():
            r = json.loads(line)
            seen[(r["run_id"], r.get("config_id"), r["problem_id"],
                  r.get("sample_index", 0))] = r
    for r in seen.values():
        # The same one-configuration rule section 3 applies. A candidate set pooled
        # across page 7's configurations would be a vote over four decoding regimes,
        # which is neither page 7's vote nor page 8's ladder and could not be compared
        # to either.
        if (r["run_id"] == P7_RUN_ID and r.get("config_id") == POOL_CONFIG
                and r.get("model_id", PRIMARY) == PRIMARY and usable(r)):
            cands[r["problem_id"]].append(r)
    cands = {k: sorted(v, key=lambda r: r.get("sample_index", 0))
             for k, v in cands.items() if len(v) >= 4}

    if not cands:
        print("page 7's pile has no problem with four or more natural-length candidates;")
        print("skipping. This needs its pile axis, not its split axis.")
    else:
        def majority(rs):
            t = Counter(r["extracted_answer"] for r in rs if r["extracted_answer"] is not None)
            return t.most_common(1)[0][0] if t else None

        plain, filtered, emptied = [], [], 0
        for pid, rs in harness.progress(sorted(cands.items()), desc="selecting", unit="item"):
            truth = rs[0]["ground_truth"]
            keep = [r for r in rs
                    if not any(check_step(s["text"])[0] == "false" for s in
                               harness.segment_steps(r["trace"], r["tokens"],
                                                     min_step_chars=MIN_STEP_CHARS))]
            if not keep:
                emptied += 1
                keep = rs                    # dropping everything is not a selection
            plain.append(bool(grade_answer(majority(rs), truth)))
            filtered.append(bool(grade_answer(majority(keep), truth)))

        SELECTION = {
            "n_problems": len(cands),
            "config_id": POOL_CONFIG,
            "page_07_run_id": P7_RUN_ID,
            "majority": mean(plain),
            "majority_after_step_filter": mean(filtered),
            "delta_pts": (mean(filtered) - mean(plain)) * 100,
            "n_problems_all_candidates_filtered": emptied,
        }
        harness.table(
            "selection, on page 7's pile",
            ["rule", "reads", "accuracy"],
            [["majority vote", "the answer", f"{SELECTION['majority']:.1%}"],
             ["majority after dropping chains with a false step", "the working",
              f"{SELECTION['majority_after_step_filter']:.1%}"]],
            note=f"{SELECTION['delta_pts']:+.1f} points over {len(cands)} problems, all from "
                 f"{POOL_CONFIG}; {emptied} had every candidate filtered and fell back to "
                 f"the full set")

        if abs(SELECTION["delta_pts"]) < 1:
            print("\nEssentially null. Say so: at this checker's recall, reading the working")
            print("buys almost nothing at selection time, and detection and localisation are")
            print("where step grading earns its keep.")

## 10. The three specimens

The page's interactive keeps its structure: the reader marks each step, then the labels are
revealed. What changes is that the chains are real and the labels are the checker's, so the
reveal is a fact rather than an attribution to a system that never ran.

| role | what it shows |
|---|---|
| `lucky` | right answer, at least one false step. Outcome grading rewards it. |
| `broken` | wrong answer, one findable break. Outcome says *wrong*; process says *step k*. |
| `clean` | right answer, nothing false. The graders agree. |

Curated by filter, shortest first. A role that cannot be filled is reported rather than faked,
and a missing role blocks translation for that panel: an authored trace must not be substituted.


In [ ]:
def pick(pool):
    ranked = sorted(pool, key=lambda r: (r["n_steps"], r["problem_id"]))
    return next((r for r in ranked if 2 <= r["n_steps"] <= 8), ranked[0] if ranked else None)


def specimen_steps(steps):
    """The steps as the page will read them.

    The PRM's per-step scores are stripped unless section 6's gate passed. Leaving
    them in would put a label on the reveal that this run just decided it could not
    stand behind, which is the exact defect the refactor exists to remove, only with a
    real model instead of a hand-written boolean. The raw scores stay in
    `traces.jsonl`, because suppressing a label is not the same as hiding the evidence.
    """
    drop = () if PRM_PUBLISHABLE else ("prm", "prm_label")
    return [{k: v for k, v in s.items() if k not in drop} for s in steps]


SPECIMENS, missing = {}, []
for role, pool in {
    "lucky":  [r for r in right if r["n_false"] > 0],
    "broken": [r for r in wrong if r["n_false"] == 1 and r["n_steps"] > 1],
    "clean":  [r for r in right if r["n_false"] == 0 and r["n_checkable"] > 0],
}.items():
    got = pick(pool)
    if got is None:
        missing.append(role)
        continue
    SPECIMENS[role] = {k: got[k] for k in (
        "problem_id", "level", "question", "ground_truth", "extracted_answer",
        "outcome_correct", "trace", "tokens", "first_false", "n_false")}
    SPECIMENS[role]["steps"] = specimen_steps(got["steps"])
    SPECIMENS[role]["role"] = role
    SPECIMENS[role]["delimiter"] = got.get("delimiter")
    SPECIMENS[role]["prm_scored"] = bool(got.get("prm_ok", False))
    SPECIMENS[role]["prm_labels_published"] = bool(got.get("prm_ok", False)
                                                   and PRM_PUBLISHABLE)
    marks = " ".join({"true": "+", "false": "X", "unknown": "?"}[s["check"]] for s in got["steps"])
    print(f"{role:<7} {got['problem_id']:<30} {got['n_steps']:>2} steps   {marks}")

print("\n  + checked true    X checked false    ? no numeric claim to check")
print(f"  step labels on the page come from: the arithmetic checker"
      + (f" and {PRM_MODEL}" if PRM_PUBLISHABLE else " only"))

if missing:
    print(f"\nMISSING ROLES: {missing}. derived.json records this. A role with no specimen")
    print("cannot be translated to the page; do not substitute an authored trace for it.")

if "broken" in SPECIMENS:
    sp = SPECIMENS["broken"]
    print(f"\n--- {sp['problem_id']}: the localisation case " + "-" * 28)
    print(textwrap.fill("Q: " + sp["question"], 92))
    for i, s in enumerate(sp["steps"]):
        arrow = "->" if s["check"] == "false" else "  "
        print(f"{arrow} {i+1}. [{s['check']:<7}] {s['text'][:84]}")
    print(f"\noutcome grader: WRONG (said {sp['extracted_answer']}, truth {sp['ground_truth']}),")
    print(f"                which condemns all {len(sp['steps'])} steps")
    print(f"process grader: step {sp['first_false'] + 1}; the {sp['first_false']} before it stand")

## 11. Artifacts

`derived.json` is the contract with the page: it reads that and nothing else, and no inference
runs at demonstration time.

The manifest records four things the other pages do not need:

- **Which grader produced which label.** A reader disputing a step label has to be able to find
  out who made it.
- **How well the model-based grader did** on the steps we could check independently, and whether
  it cleared the gate. If it did not, its labels are stripped from the specimens rather than
  footnoted, and `prm_reported_on_page` says so.
- **The chains' real provenance**, meaning page 7's run id, configuration, temperature and
  prompt when they are page 7's, and this notebook's when they are not. Two sources with two
  decoding regimes cannot share one description.
- **What the segmentation choice was worth**, so the page can quote a number knowing how much of
  it is the chains and how much is where we cut them.

In [ ]:
manifest = {
    "run_id": f"{RUN_MODE.lower()}-{ITEM_SET_HASH[:8]}",
    "notebook": "page_09_outcome_vs_process.ipynb",
    "notebook_sha256": harness.file_sha256("page_09_outcome_vs_process.ipynb"),
    "page": 9,
    "run_mode": RUN_MODE,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook_git_sha": harness.git_sha(),
    "harness": harness.fingerprint(),
    "chains": {
        # CHAIN_PROVENANCE is the source's REAL configuration, not this notebook's
        # defaults. Reused chains carry page 7's run, config, temperature and prompt;
        # chains generated here carry this notebook's. An earlier version recorded
        # this page's prompt suffix either way, which asserted a prompt that page 7's
        # chains were never shown.
        **(CHAIN_PROVENANCE or {"source": source}),
        "n": len(records),
        "filter": "truncated is false and not forced -- a chain WE ended is not the "
                  "model's reasoning, and grading it would report our cap as its error",
        "filter_audit": FILTER_AUDIT,
        "filter_caveat": "every measurement is conditional on chains the model ended "
                         "itself, which skews easier than the full population",
        "generator": PRIMARY,
        "generator_revision": harness.repo_revision(PRIMARY),
        "chain_cap": CHAIN_CAP,
    },
    "graders": {
        "outcome": "reasoning_from_scratch.ch03.grade_answer",
        "arithmetic_checker": {
            "rule": "closed-form numeric on both sides, no symbols; true/false/unknown",
            "preflight_cases": len(PREFLIGHT),
            "known_limit": "cannot see method errors that use valid arithmetic",
        },
        "prm": ({"id": PRM_MODEL, "revision": harness.repo_revision(PRM_MODEL),
                 "lineage": "same Qwen2.5 lineage as the generator, math-specialised "
                            "branch; NOT an independent family",
                 "separation_preflight": PRM_PREFLIGHT,
                 "threshold": PRM_THRESHOLD,
                 "threshold_source": RELIABILITY["threshold_source"] if RELIABILITY
                                     else "unfitted",
                 "reliability_vs_checker": RELIABILITY,
                 "reported_on_page": PRM_PUBLISHABLE,
                 "chains_not_scored": dict(PRM_FAILURES)}
                if PRM_READY else None),
    },
    "items": {"source": "MATH-500 (ch03 copy)", "filter": "level in {4, 5}",
              "n": len(items), "item_set_sha256": ITEM_SET_HASH, "seed": SEED,
              "same_as_page_7": SAME_ITEMS_AS_PAGE_7},
    "step_segmentation": {"via": "harness.step_spans", "min_step_chars": MIN_STEP_CHARS,
                          "rule": "cascade: blank lines, then lines, then sentences",
                          "delimiter_mix": dict(DELIMITER_MIX),
                          "sensitivity": SEGMENTATION_CHECK},
    "versions": {"torch": torch.__version__,
                 "transformers": __import__("transformers").__version__},
}

derived = {
    "manifest": manifest,
    "detection": DETECTION,
    "localisation": LOCALISATION,
    "selection": SELECTION,
    "prm_reliability": RELIABILITY,
    "prm_reported_on_page": PRM_PUBLISHABLE,
    "segmentation_sensitivity": SEGMENTATION_CHECK,
    "filter_audit": FILTER_AUDIT,
    "chain_provenance": CHAIN_PROVENANCE,
    "step_label_counts": dict(lab),
    "specimens": SPECIMENS,
    "missing_roles": missing,
}

DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
with TRACES_PATH.open("w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

for p, tier in ((DERIVED_PATH, "bundled"), (MANIFEST_PATH, "bundled"),
                (TRACES_PATH, "download")):
    print(f"{p.name:<16} {harness.kb(p):>9.1f} KB   <- {tier}")

## Notes for re-derivation

- The checker is **high precision, low recall by design**. Every `false` is verifiable by hand;
  every `unknown` is a step it declined to judge, not one it approved.
- Detection and localisation come from the checker alone, so they stand whether or not the PRM
  ran, and whether or not it cleared the gate. That separation is deliberate: the page's two
  load-bearing numbers do not depend on a model's opinion.
- The chains are filtered to those the model ended itself. That filter, not the source, is what
  makes them gradable; page 7's pile qualifies and its split axis does not. It is also a
  selection toward easier problems, and section 3 measures how much.
- **Only one of page 7's configurations is read.** Its dump holds the pile, the split axis, a
  diversity family at four other temperatures and a replicated greedy set. Pooling them would
  grade four decoding regimes and report the mixture as one population.
- **The PRM's threshold is fitted, not given.** It is chosen on the checker's labels and recorded
  as fitted, and the reliability beside it was measured on arithmetic steps, which are the
  friendliest ground the PRM will see here. Treat it as an upper bound.
- **The PRM is not an independent family.** Same Qwen2.5 lineage as the generator, math branch.
  It tests whether a bigger specialised checkpoint catches this one's errors, not whether an
  unrelated grader does.
- Reused chains were **sampled**, not greedy, so re-running page 7 with a different seed changes
  which chain each problem contributes. Chains generated here are greedy and still diverge
  across hardware on a few items. Expect the shape to reproduce, not every digit.

What the page may and may not claim is [`docs/page_9_refactor.md`](../docs/page_9_refactor.md),
which the translation step checks itself against
([`docs/page_6_refactor.md`](../docs/page_6_refactor.md) §6).